<a href="https://colab.research.google.com/github/maaviarizwan/skin-lesion-research/blob/main/notebooks/Stage4c_Hybrid_Fusion_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synaptix Research — Skin Lesion Classification
## Stage 4c: Hybrid V3 — Cross-Attention Feature Fusion (Colab, with Stage 3 checkpoints)

```
Stage 1 DONE  Environment + Dataset
Stage 2 DONE  Preprocessing Pipeline
Stage 3 DONE  EfficientNet-B3: F1=0.7314 | ViT-Small: F1=0.8197
Stage 4  DONE  Hybrid V1 (token transformer, from scratch): F1=0.6706 — failed, diagnosed
Stage 4c.1 DONE Hybrid V3, ImageNet-only weights (Kaggle): F1=0.8055, +TTA=0.8124
Stage 4c.2     Hybrid V3, WITH Stage 3 checkpoints  <- YOU ARE HERE
Stage 5         Paper Writing
```

## Why This Run Should Be Stronger

The Kaggle run nearly matched ViT-Small (F1=0.8055 vs 0.8197) starting both backbones from plain ImageNet weights. This run loads your **Stage 3 checkpoints directly from Drive** — `EfficientNet_B3_best.pth` and `ViT_Small_best.pth` — so both branches start already knowing skin lesion classification (F1=0.7314 and 0.8197 individually) instead of generic ImageNet features. The fusion head now only has to learn how to combine two already-strong classifiers, not also compensate for undertrained ones.

**No upload step needed** — these files already live at `/content/drive/MyDrive/Synaptix_Research/checkpoints/` from when Stage 3 ran on Colab.

**Architecture (unchanged from the Kaggle run):**
- EfficientNet-B3 -> 1536-dim summary vector
- ViT-Small -> 384-dim summary vector (CLS token)
- Both projected to 512-dim, stacked into a 2-token sequence
- Cross-attention lets each modality weigh the other per image
- Concatenated (1024-dim) -> small classifier head

**Two-phase training:** backbones frozen epochs 1-8 (fusion head only), then gently unfrozen at LR=5e-6 for epochs 9-30.

**Target to beat:** ViT-Small F1 = 0.8197 | **Expected runtime:** ~100 min on T4
---

## Cell 1 — Install + Mount Drive

In [ ]:
!pip install timm albumentations kagglehub scikit-learn tqdm -q

from google.colab import drive
drive.mount('/content/drive')
RESEARCH_DIR = '/content/drive/MyDrive/Synaptix_Research'

import torch
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Drive mounted')

## Cell 2 — All Imports + Hyperparameters

In [ ]:
import os, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    confusion_matrix, classification_report
)
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMG_SIZE   = 224
BATCH_SIZE = 16     # two backbones now share GPU memory

# Two-phase training schedule
NUM_EPOCHS      = 30
FREEZE_EPOCHS   = 8
FUSION_WARMUP   = 3
UNFREEZE_WARMUP = 3
PATIENCE        = 10

FUSION_LR   = 1e-4
BACKBONE_LR = 5e-6

LESION_NAMES = {
    'nv':'Melanocytic Nevi', 'mel':'Melanoma',
    'bkl':'Benign Keratosis', 'bcc':'Basal Cell Carcinoma',
    'akiec':'Actinic Keratoses', 'vasc':'Vascular Lesions', 'df':'Dermatofibroma'
}
print(f'Device: {device}')
print('All imports done')

## Cell 3 — Download Dataset (Session-Safe)

Colab wipes /content/ every session — re-downloads the raw images each time. Your CSVs, config, and checkpoints are safe on Drive and never need re-downloading.

In [ ]:
import kagglehub

# Paste your Kaggle token
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_c7798b0fe3d02db26a83507e2c7e334e'

print('Downloading HAM10000...')
path = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
print(f'Downloaded to: {path}')

image_path_dict = {}
for root, dirs, files in os.walk(path):
    for filename in files:
        if filename.lower().endswith(('.jpg','.jpeg','.png')):
            img_id = os.path.splitext(filename)[0]
            image_path_dict[img_id] = os.path.join(root, filename)

print(f'Images found: {len(image_path_dict)}')
if len(image_path_dict) == 0:
    raise RuntimeError('No images found. Check your token.')

## Cell 4 — Load Splits + Config from Drive

In [ ]:
with open(f'{RESEARCH_DIR}/config.json') as f:
    config = json.load(f)

label_map     = config['label_map']
class_weights = np.array(config['class_weights'])
NUM_CLASSES   = config['num_classes']
IMAGENET_MEAN = config['imagenet_mean']
IMAGENET_STD  = config['imagenet_std']
idx_to_cls    = {v: k for k, v in label_map.items()}

train_df = pd.read_csv(f'{RESEARCH_DIR}/split_train.csv')
val_df   = pd.read_csv(f'{RESEARCH_DIR}/split_val.csv')
test_df  = pd.read_csv(f'{RESEARCH_DIR}/split_test.csv')

for frame in [train_df, val_df, test_df]:
    frame['path'] = frame['image_id'].map(image_path_dict)

train_df = train_df[train_df['path'].notna()].reset_index(drop=True)
val_df   = val_df[val_df['path'].notna()].reset_index(drop=True)
test_df  = test_df[test_df['path'].notna()].reset_index(drop=True)

class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Splits loaded')

## Cell 5 — Dataset + DataLoaders

In [ ]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(0.1, 0.15, 30, p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.HueSaturationValue(10, 20, 10, p=0.4),
    A.GridDistortion(p=0.3),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()
])
val_test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()
])

class HAM10000Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.array(Image.open(row['path']).convert('RGB'))
        lbl = int(row['label'])
        if self.transform: img = self.transform(image=img)['image']
        return img, lbl

sampler = WeightedRandomSampler(
    torch.DoubleTensor([class_weights[l] for l in train_df['label']]),
    len(train_df), replacement=True
)

train_loader = DataLoader(HAM10000Dataset(train_df, train_transform),
    BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(HAM10000Dataset(val_df, val_test_transform),
    BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(HAM10000Dataset(test_df, val_test_transform),
    BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}')

## Cell 6 — Hybrid V3 + Load Stage 3 Checkpoints

Same cross-attention fusion architecture as before. This time, checkpoint paths are known exactly — they're read directly from your Drive `checkpoints/` folder, no searching needed.

In [ ]:
class HybridFusionV3(nn.Module):
    def __init__(self, num_classes=7, fusion_dim=512, nhead=8, dropout=0.2):
        super().__init__()

        self.cnn = timm.create_model('efficientnet_b3', pretrained=True,
                                     num_classes=0, global_pool='avg')
        self.vit = timm.create_model('vit_small_patch16_224', pretrained=True,
                                     num_classes=0)
        cnn_dim, vit_dim = 1536, 384

        self.cnn_proj = nn.Sequential(
            nn.Linear(cnn_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.GELU())
        self.vit_proj = nn.Sequential(
            nn.Linear(vit_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.GELU())

        self.cross_attn = nn.MultiheadAttention(
            fusion_dim, nhead, batch_first=True, dropout=dropout)
        self.attn_norm = nn.LayerNorm(fusion_dim)

        self.head = nn.Sequential(
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_classes)
        )

    def extract_features(self, x):
        return self.cnn(x), self.vit(x)

    def forward_fusion(self, cnn_feat, vit_feat):
        c = self.cnn_proj(cnn_feat)
        v = self.vit_proj(vit_feat)
        seq = torch.stack([c, v], dim=1)
        attended, _ = self.cross_attn(seq, seq, seq)
        fused = self.attn_norm(seq + attended)
        combined = fused.flatten(1)
        return self.head(combined)

    def forward(self, x):
        cnn_feat, vit_feat = self.extract_features(x)
        return self.forward_fusion(cnn_feat, vit_feat)


# ── Instantiate ──
hybrid_v3 = HybridFusionV3(num_classes=NUM_CLASSES).to(device)

# ── Load Stage 3 checkpoints directly from Drive ──
EFF_CKPT_PATH = f'{RESEARCH_DIR}/checkpoints/EfficientNet_B3_best.pth'
VIT_CKPT_PATH = f'{RESEARCH_DIR}/checkpoints/ViT_Small_best.pth'

if os.path.exists(EFF_CKPT_PATH):
    ckpt = torch.load(EFF_CKPT_PATH, map_location=device)
    result = hybrid_v3.cnn.load_state_dict(ckpt, strict=False)
    print(f'Loaded EfficientNet-B3 Stage 3 weights '
          f'(skipped {len(result.unexpected_keys)} old classifier keys)')
else:
    print(f'WARNING: {EFF_CKPT_PATH} not found — using ImageNet-only weights')

if os.path.exists(VIT_CKPT_PATH):
    ckpt = torch.load(VIT_CKPT_PATH, map_location=device)
    result = hybrid_v3.vit.load_state_dict(ckpt, strict=False)
    print(f'Loaded ViT-Small Stage 3 weights '
          f'(skipped {len(result.unexpected_keys)} old classifier keys)')
else:
    print(f'WARNING: {VIT_CKPT_PATH} not found — using ImageNet-only weights')

total = sum(p.numel() for p in hybrid_v3.parameters()) / 1e6
print(f'\nTotal parameters: {total:.1f}M')

dummy = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    out = hybrid_v3(dummy)
print(f'Output shape: {out.shape}  (should be [2, 7])')
print('Architecture verified')

## Cell 7 — Training Utilities (Two-Phase Schedule)

Phase 1 (epochs 1-8): backbones frozen under `torch.no_grad()`, fusion head only. Phase 2 (epochs 9-30): backbones gently unfrozen at LR=5e-6 to polish their already-good representations in the fused context.

In [ ]:
def get_fusion_lr(epoch, total_epochs, warmup_epochs, target_lr):
    if epoch <= warmup_epochs:
        return target_lr * epoch / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return target_lr * 0.5 * (1 + math.cos(math.pi * progress))


def get_backbone_lr(epoch, freeze_epochs, total_epochs, unfreeze_warmup, target_lr):
    if epoch <= freeze_epochs:
        return 0.0
    e = epoch - freeze_epochs
    if e <= unfreeze_warmup:
        return target_lr * e / unfreeze_warmup
    progress = (e - unfreeze_warmup) / max(1, total_epochs - freeze_epochs - unfreeze_warmup)
    return target_lr * 0.5 * (1 + math.cos(math.pi * progress))


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs   = torch.softmax(outputs, dim=1)
            loss    = criterion(outputs, labels)
            preds   = outputs.argmax(1)
            total_loss += loss.item() * images.size(0)
            correct    += preds.eq(labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return (total_loss/total, correct/total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))


def train_hybrid_v3(model, train_loader, val_loader, criterion, device,
                     epochs=NUM_EPOCHS, patience=PATIENCE,
                     freeze_epochs=FREEZE_EPOCHS,
                     fusion_warmup=FUSION_WARMUP, unfreeze_warmup=UNFREEZE_WARMUP,
                     fusion_lr=FUSION_LR, backbone_lr=BACKBONE_LR):

    fusion_params = (
        list(model.cnn_proj.parameters()) + list(model.vit_proj.parameters()) +
        list(model.cross_attn.parameters()) + list(model.attn_norm.parameters()) +
        list(model.head.parameters())
    )
    cnn_backbone_params = list(model.cnn.parameters())
    vit_backbone_params = list(model.vit.parameters())

    optimizer = torch.optim.AdamW([
        {'params': fusion_params,       'lr': 0.0},
        {'params': cnn_backbone_params, 'lr': 0.0},
        {'params': vit_backbone_params, 'lr': 0.0},
    ], weight_decay=1e-2)

    best_f1      = 0.0
    patience_ctr = 0
    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'val_f1':[]}
    ckpt_path = f'{RESEARCH_DIR}/checkpoints/Hybrid_v3_finetuned_best.pth'

    print(f'Training Hybrid V3 for up to {epochs} epochs')
    print(f'  Phase 1 (epochs 1-{freeze_epochs}): fusion head only')
    print(f'  Phase 2 (epochs {freeze_epochs+1}-{epochs}): + gentle backbone fine-tuning')
    print('-' * 80)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        backbones_frozen = epoch <= freeze_epochs

        cur_fusion_lr   = get_fusion_lr(epoch, epochs, fusion_warmup, fusion_lr)
        cur_backbone_lr = get_backbone_lr(epoch, freeze_epochs, epochs,
                                          unfreeze_warmup, backbone_lr)
        optimizer.param_groups[0]['lr'] = cur_fusion_lr
        optimizer.param_groups[1]['lr'] = cur_backbone_lr
        optimizer.param_groups[2]['lr'] = cur_backbone_lr

        for p in cnn_backbone_params: p.requires_grad = not backbones_frozen
        for p in vit_backbone_params: p.requires_grad = not backbones_frozen

        if epoch == freeze_epochs + 1:
            print(f'\n--- Unfreezing backbones at epoch {epoch} ---\n')

        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            if backbones_frozen:
                with torch.no_grad():
                    cnn_feat, vit_feat = model.extract_features(images)
                outputs = model.forward_fusion(cnn_feat, vit_feat)
            else:
                outputs = model(images)

            loss = criterion(outputs, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct    += outputs.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)

        train_loss = total_loss / total
        train_acc  = correct / total

        val_loss, val_acc, vp, vl, vprob = evaluate(model, val_loader, criterion, device)
        val_f1 = f1_score(vl, vp, average='weighted')

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        elapsed = time.time() - t0
        phase = 'frozen' if backbones_frozen else 'unfrozen'
        print(f'Epoch {epoch:02d}/{epochs} [{phase:8}] | '
              f'LR(f={cur_fusion_lr:.1e},b={cur_backbone_lr:.1e}) | '
              f'Loss {train_loss:.4f}/{val_loss:.4f} | '
              f'Acc {train_acc:.4f}/{val_acc:.4f} | '
              f'F1 {val_f1:.4f} | {elapsed:.0f}s')

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), ckpt_path)
            print(f'  --> Best F1: {best_f1:.4f} — saved')
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'Early stopping at epoch {epoch}')
                break

    print(f'\nTraining done. Best val F1: {best_f1:.4f}')
    return history, ckpt_path

print('Training utilities ready (two-phase freeze/unfreeze schedule)')

## Cell 8 — Train Hybrid V3

**Expected runtime: ~100 minutes.** Watch for the val F1 in early epochs — since both backbones already know lesion classification this time, you should see strong F1 values even during the frozen phase 1 (epochs 1-8), unlike the ImageNet-only run where phase 1 likely looked weaker.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

v3_history, v3_ckpt = train_hybrid_v3(
    hybrid_v3, train_loader, val_loader, criterion, device
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Hybrid V3 (with Stage 3 checkpoints) — Training Curves',
             fontsize=13, fontweight='bold')
ep = range(1, len(v3_history['train_loss'])+1)

axes[0].plot(ep, v3_history['train_loss'], label='Train', color='royalblue')
axes[0].plot(ep, v3_history['val_loss'],   label='Val',   color='tomato')
axes[0].axvline(x=FREEZE_EPOCHS+0.5, color='gray', linestyle=':', alpha=0.7)
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(ep, v3_history['train_acc'], label='Train', color='royalblue')
axes[1].plot(ep, v3_history['val_acc'],   label='Val',   color='tomato')
axes[1].axvline(x=FREEZE_EPOCHS+0.5, color='gray', linestyle=':', alpha=0.7)
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(ep, v3_history['val_f1'], color='green', label='Hybrid V3 (ckpt)')
axes[2].axhline(y=0.8197, color='red', linestyle='--', label='ViT-Small baseline')
axes[2].axhline(y=0.8055, color='orange', linestyle=':', label='Hybrid V3 (ImageNet-only)')
axes[2].axvline(x=FREEZE_EPOCHS+0.5, color='gray', linestyle=':', alpha=0.7)
axes[2].set_title('Val F1 vs Previous Runs (dotted vline = unfreeze point)')
axes[2].legend(fontsize=8); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(f'{RESEARCH_DIR}/figures/Hybrid_v3_finetuned_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Cell 9 — Evaluate Hybrid V3 (with checkpoints) on Test Set

In [ ]:
hybrid_v3.load_state_dict(torch.load(v3_ckpt, map_location=device))

_, _, v3_preds, v3_labels, v3_probs = evaluate(
    hybrid_v3, test_loader, criterion, device
)

v3_acc = accuracy_score(v3_labels, v3_preds)
v3_f1  = f1_score(v3_labels, v3_preds, average='weighted')
try:
    v3_auc = roc_auc_score(v3_labels, v3_probs, multi_class='ovr', average='macro')
except: v3_auc = 0.0

print('Hybrid V3 (with Stage 3 checkpoints) — Test Set Results:')
print('=' * 55)
print(f'  Accuracy  : {v3_acc:.4f}')
print(f'  F1 (W)    : {v3_f1:.4f}')
print(f'  AUC-ROC   : {v3_auc:.4f}')
print(f'  vs ViT F1                  : {v3_f1 - 0.8197:+.4f}')
print(f'  vs Hybrid V3 (ImageNet-only): {v3_f1 - 0.8055:+.4f}')

print('\nPer-class Report:')
print(classification_report(v3_labels, v3_preds,
    target_names=list(label_map.keys())))

cm = confusion_matrix(v3_labels, v3_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=list(label_map.keys()),
    yticklabels=list(label_map.keys()))
plt.title('Hybrid V3 (with checkpoints) — Confusion Matrix', fontweight='bold')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{RESEARCH_DIR}/figures/Hybrid_v3_finetuned_confusion.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Cell 9b — Bonus: Test-Time Augmentation (TTA)

Average predictions from the original and horizontally-flipped image. Free accuracy boost, no retraining needed.

In [ ]:
def evaluate_tta(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            probs_orig = torch.softmax(model(images), dim=1)
            flipped    = torch.flip(images, dims=[3])
            probs_flip = torch.softmax(model(flipped), dim=1)
            avg_probs  = (probs_orig + probs_flip) / 2
            preds = avg_probs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(avg_probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


tta_preds, tta_labels, tta_probs = evaluate_tta(hybrid_v3, test_loader, device)

v3_tta_acc = accuracy_score(tta_labels, tta_preds)
v3_tta_f1  = f1_score(tta_labels, tta_preds, average='weighted')
try:
    v3_tta_auc = roc_auc_score(tta_labels, tta_probs, multi_class='ovr', average='macro')
except: v3_tta_auc = 0.0

print('Hybrid V3 (checkpoints) + TTA — Test Set Results:')
print('=' * 55)
print(f'  Accuracy  : {v3_tta_acc:.4f}  (raw: {v3_acc:.4f}, change: {v3_tta_acc-v3_acc:+.4f})')
print(f'  F1 (W)    : {v3_tta_f1:.4f}  (raw: {v3_f1:.4f}, change: {v3_tta_f1-v3_f1:+.4f})')
print(f'  AUC-ROC   : {v3_tta_auc:.4f}  (raw: {v3_auc:.4f}, change: {v3_tta_auc-v3_auc:+.4f})')

## Cell 10 — Final Comparison

This is **Table 1** for your paper — the headline result.

In [ ]:
EFF_ACC, EFF_F1, EFF_AUC = 0.7053, 0.7314, 0.9620
VIT_ACC, VIT_F1, VIT_AUC = 0.8044, 0.8197, 0.9763
V1_ACC,  V1_F1,  V1_AUC  = 0.6367, 0.6706, 0.9398
V3_IN_ACC, V3_IN_F1, V3_IN_AUC = 0.7878, 0.8055, 0.9733   # ImageNet-only run

print('=' * 80)
print('  FINAL RESULTS — TABLE 1 (for paper)')
print('=' * 80)
print(f'{"Model":<32} {"Accuracy":>10} {"F1 (W)":>10} {"AUC-ROC":>10}')
print('-' * 66)
print(f'{"EfficientNet-B3":<32} {EFF_ACC:>10.4f} {EFF_F1:>10.4f} {EFF_AUC:>10.4f}')
print(f'{"ViT-Small":<32} {VIT_ACC:>10.4f} {VIT_F1:>10.4f} {VIT_AUC:>10.4f}')
print(f'{"Hybrid V1 (token-fusion)":<32} {V1_ACC:>10.4f} {V1_F1:>10.4f} {V1_AUC:>10.4f}')
print(f'{"Hybrid V3 (ImageNet-only)":<32} {V3_IN_ACC:>10.4f} {V3_IN_F1:>10.4f} {V3_IN_AUC:>10.4f}')
print(f'{"Hybrid V3 (Stage3 ckpts)":<32} {v3_acc:>10.4f} {v3_f1:>10.4f} {v3_auc:>10.4f}')
print(f'{"Hybrid V3 (ckpts) + TTA":<32} {v3_tta_acc:>10.4f} {v3_tta_f1:>10.4f} {v3_tta_auc:>10.4f}')
print('=' * 80)

best_base_f1 = max(EFF_F1, VIT_F1)
print(f'  Best result vs best baseline (ViT) : {max(v3_f1,v3_tta_f1) - best_base_f1:+.4f} F1')
print(f'  vs Hybrid V3 ImageNet-only run      : {v3_f1 - V3_IN_F1:+.4f} F1')
if max(v3_f1, v3_tta_f1) > best_base_f1:
    print('  Hybrid V3 OUTPERFORMS both individual baselines!')

models = ['EfficientNet-B3', 'ViT-Small', 'Hybrid V1', 'Hybrid V3\n(ImageNet)',
          'Hybrid V3\n(ckpts)', 'Hybrid V3\n(ckpts)+TTA']
accs   = [EFF_ACC, VIT_ACC, V1_ACC, V3_IN_ACC, v3_acc, v3_tta_acc]
f1s    = [EFF_F1,  VIT_F1,  V1_F1,  V3_IN_F1,  v3_f1,  v3_tta_f1]
aucs   = [EFF_AUC, VIT_AUC, V1_AUC, V3_IN_AUC, v3_auc, v3_tta_auc]
colors = ['#4C72B0', '#DD8452', '#999999', '#fbbf24', '#2ecc71', '#16a34a']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Full Model Comparison — Test Set', fontsize=14, fontweight='bold')

for ax, vals, title in zip(axes, [accs, f1s, aucs],
                           ['Accuracy', 'F1 (Weighted)', 'AUC-ROC']):
    bars = ax.bar(models, vals, color=colors, edgecolor='black', linewidth=0.8)
    ax.set_ylim(0.6, 1.02)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=20, labelsize=8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.4f}', ha='center', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{RESEARCH_DIR}/figures/final_comparison_v3_complete.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved')

## Cell 11 — Save All Results to Drive

In [ ]:
results = {
    'EfficientNet_B3':           {'accuracy': EFF_ACC, 'f1_weighted': EFF_F1, 'auc_roc': EFF_AUC},
    'ViT_Small':                 {'accuracy': VIT_ACC, 'f1_weighted': VIT_F1, 'auc_roc': VIT_AUC},
    'Hybrid_V1':                 {'accuracy': V1_ACC,  'f1_weighted': V1_F1,  'auc_roc': V1_AUC},
    'Hybrid_V3_ImageNetOnly':    {'accuracy': V3_IN_ACC, 'f1_weighted': V3_IN_F1, 'auc_roc': V3_IN_AUC},
    'Hybrid_V3_Stage3Checkpoints': {'accuracy': v3_acc, 'f1_weighted': v3_f1, 'auc_roc': v3_auc,
                                    'history': v3_history},
    'Hybrid_V3_Checkpoints_TTA': {'accuracy': v3_tta_acc, 'f1_weighted': v3_tta_f1, 'auc_roc': v3_tta_auc}
}

with open(f'{RESEARCH_DIR}/results/stage4c_final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Saved to Drive:')
print(f'  checkpoints/Hybrid_v3_finetuned_best.pth')
print(f'  results/stage4c_final_results.json')
print(f'  figures/final_comparison_v3_complete.png')
print(f'  figures/Hybrid_v3_finetuned_curves.png')
print(f'  figures/Hybrid_v3_finetuned_confusion.png')

## Cell 12 — Stage 4c Complete

In [ ]:
print('=' * 70)
print('  STAGE 4c COMPLETE — FINAL HYBRID RESULTS')
print('=' * 70)
print()
print('  Model                          Acc      F1       AUC')
print('  ──────────────────────────────────────────────────────')
print(f'  EfficientNet-B3              {EFF_ACC:.4f}  {EFF_F1:.4f}  {EFF_AUC:.4f}')
print(f'  ViT-Small                    {VIT_ACC:.4f}  {VIT_F1:.4f}  {VIT_AUC:.4f}')
print(f'  Hybrid V1 (token, failed)    {V1_ACC:.4f}  {V1_F1:.4f}  {V1_AUC:.4f}')
print(f'  Hybrid V3 (ImageNet-only)    {V3_IN_ACC:.4f}  {V3_IN_F1:.4f}  {V3_IN_AUC:.4f}')
print(f'  Hybrid V3 (Stage3 ckpts)     {v3_acc:.4f}  {v3_f1:.4f}  {v3_auc:.4f}')
print(f'  Hybrid V3 (ckpts) + TTA      {v3_tta_acc:.4f}  {v3_tta_f1:.4f}  {v3_tta_auc:.4f}')
print()
print(f'  Best vs best baseline F1: {max(v3_f1,v3_tta_f1) - max(EFF_F1,VIT_F1):+.4f}')
print()
print('  NEXT: Stage 5 — Paper Writing')
print('  Target venues: MIDL, ISBI')
print('=' * 70)